# CT.M1 – Data Acquisition
### Notebook 03. ChEMBL Data Retrieval 

**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/03_chembl_data_retrieval.ipynb)


---


### Contents

This notebook mirrors the structure of the PubChem notebook, but using **ChEMBL**. We will walk through the "Data Journey" of a medicinal chemist:
- The Search: Finding a specific protein target (e.g., an enzyme or receptor).
- The Extraction: Retrieving all experimental bioactivity data associated with that target.
- The Refinement: Filtering results to ensure data quality (e.g., standardizing units to $nM$).
- The Visualization: Rendering chemical structures directly in our notebook.

**Key idea:** 
ChEMBL is bioactivity-centered (assays/targets/activities). You’ll typically retrieve:
- Molecules (structures + computed properties)
- Activities ($IC_{50}$, $EC_{50}$, $K_i$, etc.) 
- Assays and Targets (biological context)

This notebook is designed to be completed in approximately **120–180 minutes**.


1. Step 0 – Computational environment
2. Step 1 – Defining the search space (natural language → ChEMBL IDs)
3. Step 2 – Retrieving molecule records (SMILES, InChIKey, basic properties)
4. Step 3 – QC and initial inspection
5. Step 4 – Optional: retrieve bioactivities for your molecules
6. Step 5 – Save dataset

---

### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourName_YourID_03_chembl_data_retrieval.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---


## ChEMBL Data Retrieval  

Programmatic access to bioactivity databases is a cornerstone of modern Cheminformatics. While manual web searches are useful for looking up a single molecule, programmatic retrieval allows us to build large, structured datasets for machine learning, SAR (Structure-Activity Relationship) analysis, and drug discovery pipelines.

**What is ChEMBL?**
**ChEMBL** is an open-source, manually curated database maintained by the European Molecular Biology Laboratory (EMBL-EBI). It is unique because its data is primarily extracted from medicinal chemistry literature (scientific journals), where experts capture:
* Molecular Structures: 2D and 3D representations of **bioactive compounds**.
* Biological Targets: The specific **proteins or cell lines** the compounds interact with.
* Bioactivity Measurements: Quantitative data such as $IC_{50}$, $K_i$, and $EC_{50}$ values.


**What is the difference from PubChem?**
If you search for **“flavonoids”** in PubChem, you will retrieve a massive list of chemical structures. PubChem is a **structure-centered database**, which means searches are primarily based on molecular identifiers such as SMILES, InChI, or InChIKey. Its focus is the **chemical universe**: structures, calculated properties, and associated records.

In contrast, if you search for flavonoids in **ChEMBL**, you will retrieve flavonoids that are linked to **validated experimental bioactivity data** (e.g., “Flavonoid X inhibits Enzyme Y with an $IC_{50}$ of 10 nM”). This means that ChEMBL is fundamentally **target-centric** and **bioactivity-oriented**.

In ChEMBL, we are not simply interested in a large number of flavonoid structures. Instead, we are interested in identifying **which flavonoids are biologically active against a specific target protein**, under defined experimental conditions.

You can think of ChEMBL as representing a **biological space**, where molecules are connected to assays, targets, activity measurements, and pharmacological profiles. For this reason, ChEMBL is considered a gold standard resource for building Structure–Activity Relationship (SAR) models, because it provides curated, normalized, and target-linked bioactivity data suitable for quantitative analysis and machine learning.

---

### Step 0: The Computational Environment

To interact with the **ChEMBL** database, we will use the `requests` library. This allows us to send **HTTP GET** requests to the ChEMBL REST API and receive data in **JSON** format.

Using `requests` directly (instead of a specialized library) helps us understand the underlying data structure of the database. We will also use `pandas` to transform the raw JSON response into a clean, searchable table.

In [ ]:
# Step 0

import os
import time
import requests
import pandas as pd

print("Environment configured successfully.")
print(f"Requests version: {requests.__version__}")


### Step 1: Defining the Chemical Search Space

In the **ChEMBL** ecosystem, a common entry point is a text-based search (e.g., "Aspirin" or "Caffeine"). This query must be translated into a unique identifier known as the **molecule_chembl_id** via the `/molecule/search` endpoint.

Key **API** Concepts:
* The Identifier: The `molecule_chembl_id` is the "primary key" that links a chemical structure to all its biological assays and properties across the database.
* Pagination Strategy: To prevent server overload and slow response times, ChEMBL uses **pagination**. Instead of sending thousands of results at once, the API serves data in "chunks" using two parameters:
    * `limit`: The maximum number of records to return in a single request (e.g., 20).
    * `offset`: The number of records to skip to reach the next "page" of results.


> **Notice:** Think of the ChEMBL database as a massive encyclopedia. ChEMBL responses are **paginated** using `limit` and `offset`. The `limit` defines how many rows you can see on one page, while the `offset` tells the API how many rows you have already read.

In [ ]:
# Step 1

# Define the API entry point
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

# Setup a local data directory
OUT_DIR = os.path.join("..", "data_raw")
os.makedirs(OUT_DIR, exist_ok=True)

# Search configuration
QUERY = "flavonoid"      # Try: "aspirin", "capsaicin", "cannabinoid", "kinase inhibitor"
LIMIT = 50               # items per page (keep small for tutorials)
MAX_RESULTS = 200        # safety cap to avoid pulling too much in a tutorial

print(f"Query term: '{QUERY}'")
print(f"ChEMBL API endpoint: {BASE_URL}")
print(f"Local storage directory: {OUT_DIR}")


### Step 2: Exploring Different Search Strategies in ChEMBL

In the previous step, we looked at a basic search. However, ChEMBL is a **relational database**, meaning its true power lies in how different biological and chemical entities are linked.

Depending on your research goal, you might not want to search by a "name." Instead, you might want to find molecules based on how they work or what they target. In this cell, we explore four professional search strategies:

1. **Mechanism of Action (MoA)**
* The Question: "Find me all drugs that act as an agonist (activator)."
* Value: Useful for identifying functional classes of drugs regardless of their chemical structure.

2. **ATC Classification**
* The Question: "Show me drugs classified by their therapeutic use (e.g., Nervous System vs. Cardiovascular)."
* Value: This is the standard used by the World Health Organization (WHO) to categorize approved medicines.

3. **Target Filtering (By Type)**
* The Question: "Give me a list of Single Proteins that have been studied."
* Value: Helps narrow down the search space to high-quality biological targets, excluding complex "cell lines" or "tissues" that might yield "noisy" data.

4. **Free-Text Molecule Search**
* The Question: "Find molecules related to the phrase kinase inhibitor."
* Value: A broad "catch-all" search useful when you are in the early discovery phase and don't have a specific ID yet.

>**Notice:** Observe the use of `__icontains` in the URL parameters. This is a common API pattern that allows for "fuzzy" matching—searching for a word inside a string without worrying about exact case sensitivity.

In [ ]:
# Step 2 

from urllib.parse import urlencode

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def build_url(endpoint, params=None):
    """Utility function to construct a full ChEMBL API URL."""
    if params:
        return f"{BASE_URL}/{endpoint}?{urlencode(params)}"
    return f"{BASE_URL}/{endpoint}"

# 1. Search by Mechanism of Action (MoA)
mechanism_example = build_url(
    "mechanism.json",
    {"mechanism_of_action__icontains": "agonist", "limit": 20}
)

# 2.Search by ATC Classification (approved drugs)
atc_example = build_url(
    "atc_class.json",
    {"limit": 20}
)

# 3. Search Targets (e.g., target family or type)
target_example = build_url(
    "target.json",
    {"target_type": "SINGLE PROTEIN", "limit": 20}
)

# 4. Free Text Molecule Search (less structured)
molecule_search_example = build_url(
    "molecule/search.json",
    {"q": "kinase inhibitor", "limit": 20}
)

print("Different ChEMBL Search Strategies:\n")
print("1. Mechanism of Action search:")
print(mechanism_example, "\n")

print("2. ATC Classification search:")
print(atc_example, "\n")

print("3. Target search:")
print(target_example, "\n")

print("4. Free-text molecule search:")
print(molecule_search_example)


### Step 2.1 – `/mechanism.json`

Unlike PubChem, which is structure-centered, ChEMBL allows us to explore molecules through their **mechanism of action (MoA)**.

The `/mechanism.json` endpoint links molecules to:
- Their annotated **mechanism of action**
- The corresponding **target**
- The pharmacological **action type** (e.g., AGONIST, ANTAGONIST, INHIBITOR)

In this step, we explore molecules annotated as **agonists**.

**Important semantic detail:** 
If we use a substring search such as: `mechanism_of_action__icontains = "agonist"` the API will also return entries containing *"antagonist"*, since the word *agonist* appears inside it. 

To avoid this ambiguity, we:
1. Retrieve a broad set of results using `icontains`
2. Apply a **post-query filter** locally:
   - Preferably using the structured field `action_type`
   - Alternatively using a word-boundary regex to exclude "antagonist"

This approach highlights an important principle in bioinformatics databases:

> When a structured field exists (e.g., `action_type`), it is more reliable than free-text matching.

Now we will query `/mechanism.json` and filter the results to isolate true agonists.


In [ ]:
# Step 2.1 – /mechanism.json

import requests
import pandas as pd
import re
from urllib.parse import urlencode

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_get(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    endpoint = endpoint.lstrip("/")
    url = f"{BASE_URL}/{endpoint}"
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

# 1) Broad query (substring) to collect candidates
TERM = "agonist"
params = {
    "mechanism_of_action__icontains": TERM,
    "limit": 100  # increase if you want a larger sample
}

url = f"{BASE_URL}/mechanism.json?{urlencode(params)}"
print("Requesting (broad):")
print(url)

data = chembl_get("mechanism.json", params=params)
rows = data.get("mechanisms", [])
print("\nRows returned (broad):", len(rows))

df = pd.json_normalize(rows)

# 2A) Best filter (structured): keep only action_type == AGONIST (if present)
if "action_type" in df.columns:
    df["action_type"] = df["action_type"].astype(str)
    df_struct = df[df["action_type"].str.upper().eq("AGONIST")].copy()
else:
    df_struct = pd.DataFrame()

# 2B) Fallback filter (text): match whole word "agonist" but NOT "antagonist"
# word-boundary + negative lookbehind for "ant"
if "mechanism_of_action" in df.columns:
    moa = df["mechanism_of_action"].fillna("").astype(str)
    mask_word = moa.str.contains(r"\bagonist\b", flags=re.IGNORECASE, regex=True)
    mask_not_ant = ~moa.str.contains(r"\bantagonist\b", flags=re.IGNORECASE, regex=True)
    df_text = df[mask_word & mask_not_ant].copy()
else:
    df_text = pd.DataFrame()

print("\nFiltered counts:")
print("• action_type == AGONIST:", len(df_struct))
print("• text contains 'agonist' but not 'antagonist':", len(df_text))

# Choose the cleanest available filtered table for preview
df_final = df_struct if len(df_struct) > 0 else df_text

# Preview
keep = [
    "molecule_chembl_id",
    "mechanism_of_action",
    "action_type",
    "target_chembl_id",
    "target_pref_name",
]
keep = [c for c in keep if c in df_final.columns]

df_final[keep].head(10) if not df_final.empty else df_final.head()


### Step 2.2 – `/atc_class` (JSON output)

ChEMBL also provides access to **ATC (Anatomical Therapeutic Chemical) classification**, a hierarchical system used to categorize approved drugs according to:

- The organ or system they act upon
- Their therapeutic use
- Their pharmacological and chemical properties

The `/atc_class` endpoint allows us to explore this therapeutic hierarchy.

Unlike molecule or mechanism endpoints, ATC entries are organized in **five hierarchical levels**:

- **Level 1** → Anatomical main group (e.g., "A" – Alimentary tract and metabolism)
- **Level 2–4** → Increasingly specific therapeutic/pharmacological subgroups
- **Level 5** → Chemical substance level

In this step, we:

1. Request a small sample of ATC entries.
2. Inspect the JSON structure returned by the API.
3. Extract the hierarchical levels into a structured table.

**Important note:**
ATC classification applies mainly to **approved drugs**, not to all bioactive molecules in ChEMBL. Therefore, this endpoint represents a **therapeutic space**, rather than the broader bioactivity space explored with `/mechanism.json`.

This illustrates another key concept:

> ChEMBL connects molecules not only to targets and mechanisms, but also to formal therapeutic classifications when applicable.

In [ ]:
# Step 2.2 – /atc_class

import requests
import pandas as pd
from urllib.parse import urlencode

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_get_json(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    endpoint = endpoint.lstrip("/")
    url = f"{BASE_URL}/{endpoint}"
    headers = {"Accept": "application/json"}
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

# Small sample (ATC is hierarchical; this returns entries across levels)
params = {"limit": 20, "offset": 0}

url = f"{BASE_URL}/atc_class?{urlencode(params)}"
print("Requesting:")
print(url)

data = chembl_get_json("atc_class", params=params)

# The returned list is under the key "atc"
rows = data.get("atc", [])
print("\nRows returned:", len(rows))

df_atc = pd.json_normalize(rows)

cols = [
    "level1", "level1_description",
    "level2", "level2_description",
    "level3", "level3_description",
    "level4", "level4_description",
    "level5", "level5_description",
]
cols = [c for c in cols if c in df_atc.columns]

df_atc[cols].head(10) if cols else df_atc.head(10)


### Step 2.3 – `/target` (JSON output)

One of the core strengths of ChEMBL is its **target-centric organization**.

While PubChem is primarily structure-oriented, ChEMBL connects molecules to **biological targets**, allowing us to explore pharmacology from the perspective of proteins and biological systems.

The `/target` endpoint allows us to retrieve information about:

- Individual proteins (e.g., enzymes, receptors, kinases)
- Protein families
- Complexes
- Organisms associated with each target

Each target entry typically includes:

- `target_chembl_id` → unique ChEMBL identifier
- `pref_name` → preferred name of the target
- `organism` → species
- `target_type` → classification (e.g., SINGLE PROTEIN, PROTEIN FAMILY)
- Additional metadata such as confidence scores

In this step, we:

1. Query targets by `target_type` (e.g., "SINGLE PROTEIN").
2. Optionally apply a keyword filter (e.g., "kinase", "PPAR", "receptor").
3. Inspect the structured biological entities returned by the API.

This step is crucial conceptually:

> Instead of asking “Which molecules exist?”, we now ask  
> “Which biological targets exist, and how are they classified?”

Understanding the target layer prepares us for the next logical step in ChEMBL workflows:

- Retrieving bioactivity data using `/activity`
- Building target-centric datasets
- Constructing SAR and machine learning models

In [ ]:
# Step 2.3 – /target 

import requests
import pandas as pd
from urllib.parse import urlencode

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_get_json(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    endpoint = endpoint.lstrip("/")
    url = f"{BASE_URL}/{endpoint}"
    headers = {"Accept": "application/json"}
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

# --- Choose a target exploration mode ---
TARGET_TYPE = "SINGLE PROTEIN"   # Common types: "SINGLE PROTEIN", "PROTEIN FAMILY", etc.
KEYWORD = "kinase"              # Optional: try "PPAR", "GPCR", "dopamine", "interleukin"
USE_KEYWORD_FILTER = True       # Set False to just sample targets by type

params = {
    "target_type": TARGET_TYPE,
    "limit": 20,
    "offset": 0
}

# Optional keyword filter (field name can vary; we use a safe approach via pref_name)
if USE_KEYWORD_FILTER:
    params["pref_name__icontains"] = KEYWORD

url = f"{BASE_URL}/target?{urlencode(params)}"
print("Requesting:")
print(url)

data = chembl_get_json("target", params=params)

# Target list is typically under the key "targets"
# (We still print keys for transparency)
print("\nTop-level keys:", list(data.keys()))

rows = data.get("targets", [])
print("Rows returned:", len(rows))

df_targets = pd.json_normalize(rows)

cols = [
    "target_chembl_id",
    "pref_name",
    "organism",
    "target_type",
    "confidence_score",
    "tax_id",
]
cols = [c for c in cols if c in df_targets.columns]

df_targets[cols].head(10) if cols else df_targets.head(10)

### Step 2.4 – `/molecule/search`

The `/molecule/search` endpoint allows us to perform a **free-text query** across molecule records in ChEMBL.

This type of search is conceptually similar to PubChem’s text-based search, where a query term (e.g., "kinase inhibitor") is matched against molecule-associated metadata.

In this step, we:

1. Define a free-text query (`q` parameter).
2. Retrieve a limited number of results (`limit`, `offset`).
3. Inspect the list of matching molecules.

Each result typically includes:

- `molecule_chembl_id` → unique identifier
- `pref_name` → preferred compound name (if available)
- `molecule_type` → small molecule, protein, etc.
- `max_phase` → highest clinical phase reached (if applicable)

**Important conceptual distinction:**
Unlike `/mechanism`, `/target`, or `/atc_class`, this endpoint is **not structured around biological relationships**.  
It performs a broader textual search and may return heterogeneous results.

This reinforces an important idea:

> `/molecule/search` explores chemical entities through text matching,  
> whereas other endpoints explore structured pharmacological or biological relationships.

In practice, `/molecule/search` is useful for:
- Quickly identifying known compounds
- Exploring molecules associated with a term
- Performing preliminary exploration before applying structured filters

However, for building robust pharmacological datasets, structured endpoints such as `/target` and `/activity` are preferred.

In [ ]:
# Step 2.4 – /molecule/search

import requests
import pandas as pd
from urllib.parse import urlencode

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_get_json(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    endpoint = endpoint.lstrip("/")
    url = f"{BASE_URL}/{endpoint}"
    headers = {"Accept": "application/json"}
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

# --- Define free-text query ---
QUERY = "kinase inhibitor"   # Try: "PPAR", "dopamine receptor", "antibiotic"

params = {
    "q": QUERY,
    "limit": 20,
    "offset": 0
}

url = f"{BASE_URL}/molecule/search?{urlencode(params)}"
print("Requesting:")
print(url)

data = chembl_get_json("molecule/search", params=params)

print("\nTop-level keys:", list(data.keys()))

# Results are stored under "molecules"
rows = data.get("molecules", [])
print("Rows returned:", len(rows))

df_mol = pd.json_normalize(rows)

cols = [
    "molecule_chembl_id",
    "pref_name",
    "molecule_type",
    "max_phase",
]
cols = [c for c in cols if c in df_mol.columns]

df_mol[cols].head(10) if cols else df_mol.head(10)

### Step 2.5 – `/activity`

The `/activity` endpoint represents the core of **ChEMBL’s biological value**.

Unlike structural endpoints, `/activity` provides access to **quantitative experimental measurements**, such as:

- IC₅₀ (half maximal inhibitory concentration)
- EC₅₀ (half maximal effective concentration)
- Kᵢ (inhibition constant)
- Kd (dissociation constant)
- pChEMBL (log-transformed potency values)

Each activity record connects three fundamental entities:

- **Molecule** (`molecule_chembl_id`)
- **Target** (`target_chembl_id`)
- **Assay** (`assay_chembl_id`)

This creates a structured relationship: `molecule → assay → target → quantitative activity`


In this step, we:

1. Select a `target_chembl_id`.
2. Filter for a specific measurement type (e.g., `"IC50"`).
3. Retrieve a limited number of activity records.
4. Inspect standardized fields such as:
   - `standard_value`
   - `standard_units`
   - `pchembl_value`
   - `standard_relation`


**Important conceptual point:**
This endpoint transforms ChEMBL from a molecular registry into a **quantitative pharmacological dataset**.

While `/molecule/search` explores chemical entities and `/target` explores biological entities,  
`/activity` connects both through experimentally measured potency values.

This is why ChEMBL is considered a gold-standard resource for:

- Structure–Activity Relationship (SAR) analysis
- Quantitative modeling
- Machine learning in drug discovery
- Target-centric dataset construction


In [ ]:
# Step 2.5 – /activity

import requests
import pandas as pd
from urllib.parse import urlencode

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_get_json(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    endpoint = endpoint.lstrip("/")
    url = f"{BASE_URL}/{endpoint}"
    headers = {"Accept": "application/json"}
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

# --- Define target of interest ---
# You can replace this with any target_chembl_id retrieved previously
TARGET_CHEMBL_ID = "CHEMBL203"   # Example: EGFR

params = {
    "target_chembl_id": TARGET_CHEMBL_ID,
    "standard_type": "IC50",   # Focus on quantitative inhibition values
    "limit": 20,
    "offset": 0
}

url = f"{BASE_URL}/activity?{urlencode(params)}"
print("Requesting:")
print(url)

data = chembl_get_json("activity", params=params)

print("\nTop-level keys:", list(data.keys()))

# Activities are stored under "activities"
rows = data.get("activities", [])
print("Rows returned:", len(rows))

df_activity = pd.json_normalize(rows)

# Select key quantitative columns if present
columns_to_show = [
    "molecule_chembl_id",
    "target_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "standard_flag"
]

columns_to_show = [c for c in columns_to_show if c in df_activity.columns]

df_activity[columns_to_show].head(10) if columns_to_show else df_activity.head(10)

### Step 3 – From Free-Text Query to `molecule_chembl_id` (Paginated Retrieval)

In this step, we move from exploratory API calls to a structured data-retrieval workflow.

Instead of retrieving a small preview (e.g., 20 rows), we implement a **paginated search strategy** to systematically collect molecule identifiers from ChEMBL.

Why is this necessary?

ChEMBL (like most modern APIs) uses **pagination**, meaning that:
- Results are returned in batches (`limit`)
- Additional results require increasing the `offset`
- Large result sets must be retrieved iteratively

The function `search_molecules()` performs the following operations:

1. Sends repeated requests to `/molecule/search`
2. Uses `limit` and `offset` to navigate across result pages
3. Collects `molecule_chembl_id` values
4. Stops when:
   - The API returns fewer rows than the limit (end of results), or
   - The predefined `max_results` threshold is reached
5. Removes duplicate IDs while preserving order

This is a crucial step conceptually:

> We are no longer exploring the API manually.  
> We are constructing a reproducible data extraction pipeline.

The output of this step is a clean list of unique `molecule_chembl_id`, which can then be used to:

- Retrieve molecular properties (`/molecule`)
- Retrieve bioactivities (`/activity`)
- Build SAR datasets
- Construct machine learning-ready tables

---

### Why This Matters

This step transforms a free-text query (e.g., `"flavonoid"`) into a structured dataset backbone.

In practical terms: `text query → molecule_chembl_id list → structured dataset`.

This is the first step toward building a target-centric or molecule-centric bioactivity dataset using ChEMBL.


In [ ]:
# Step 3

import time
import requests
from urllib.parse import quote

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

QUERY = "antibiotic"
LIMIT = 50
MAX_RESULTS = 200


def chembl_get(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    """Helper function for GET requests to ChEMBL (forces JSON)."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    headers = {"Accept": "application/json"}
    response = requests.get(url, params=params, headers=headers, timeout=timeout)
    response.raise_for_status()
    return response.json()

def search_molecules(query: str, limit: int = 20, max_results: int = 200) -> list[str]:
    """Search molecules by text query and return list of molecule_chembl_id (paginated)."""
    offset = 0
    molecule_ids = []

    while True:
        data = chembl_get(
            "molecule/search",
            params={"q": query, "limit": limit, "offset": offset}
        )
        molecules = data.get("molecules", [])

        if offset == 0:
            print(f"Example endpoint: {BASE_URL}/molecule/search?q={quote(query)}&limit={limit}&offset={offset}")

        for molecule in molecules:
            chembl_id = molecule.get("molecule_chembl_id")
            if chembl_id:
                molecule_ids.append(chembl_id)

        if len(molecules) < limit or len(molecule_ids) >= max_results:
            break

        offset += limit
        time.sleep(0.1)  # polite delay

    # Remove duplicates while preserving order
    seen = set()
    unique_ids = []
    for cid in molecule_ids:
        if cid not in seen:
            seen.add(cid)
            unique_ids.append(cid)

    return unique_ids[:max_results]

# Execute search
molecule_ids = search_molecules(QUERY, limit=LIMIT, max_results=MAX_RESULTS)
print(f"Total molecule_chembl_id retrieved: {len(molecule_ids)}")
print(f"First 10: {molecule_ids[:10]}")


### Step 4 – From `molecule_chembl_id` to Structured Molecular Records

Now that we have obtained a clean list of unique `molecule_chembl_id`, we can move to the next logical step:

Retrieving structured molecular records from ChEMBL.

Each `molecule_chembl_id` acts as a stable identifier that allows us to request:

- Canonical SMILES
- InChI / InChIKey
- Molecular properties (MW, AlogP, TPSA, etc.)
- Clinical phase information
- Molecule type

At this stage, we transition from: `Text-based exploration → Identifier collection`

to: `Identifier list → Structured molecular dataset`

---

This separation is important from a data engineering perspective:
- Step 3 builds the backbone (ID list)
- Step 4 enriches the backbone with structured chemical information
- Later steps will connect molecules to biological activity

This modular workflow ensures that our dataset construction is:
- Reproducible
- Transparent
- Scalable
- Suitable for downstream SAR or machine learning analysis

In the next cell, we will retrieve molecular records in batches using the `/molecule` endpoint.


In [ ]:
# Step 4 – Retrieve structured molecular records using /molecule

import time
import requests
import pandas as pd

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_get(endpoint: str, params: dict | None = None, timeout: int = 30) -> dict:
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    headers = {"Accept": "application/json"}
    response = requests.get(url, params=params, headers=headers, timeout=timeout)
    response.raise_for_status()
    return response.json()

def fetch_molecules_bulk(molecule_ids: list[str], batch_size: int = 50) -> pd.DataFrame:
    """
    Retrieve molecule records in batches using molecule_chembl_id__in filter.
    """
    all_rows = []

    for i in range(0, len(molecule_ids), batch_size):
        batch = molecule_ids[i:i+batch_size]

        params = {
            "molecule_chembl_id__in": ",".join(batch),
            "limit": batch_size,
            "offset": 0
        }

        data = chembl_get("molecule", params=params)
        molecules = data.get("molecules", [])

        all_rows.extend(molecules)
        time.sleep(0.1)  # polite delay

    df = pd.json_normalize(all_rows)

    # Select key columns if present
    columns_to_keep = [
        "molecule_chembl_id",
        "pref_name",
        "molecule_structures.canonical_smiles",
        "molecule_structures.standard_inchi_key",
        "molecule_properties.full_mwt",
        "molecule_properties.alogp",
        "molecule_properties.tpsa",
        "max_phase",
        "molecule_type"
    ]

    columns_to_keep = [c for c in columns_to_keep if c in df.columns]

    df_clean = df[columns_to_keep].rename(columns={
        "molecule_structures.canonical_smiles": "SMILES",
        "molecule_structures.standard_inchi_key": "InChIKey",
        "molecule_properties.full_mwt": "MolecularWeight",
        "molecule_properties.alogp": "AlogP",
        "molecule_properties.tpsa": "TPSA",
    })

    return df_clean

# Execute bulk retrieval
df_molecules = fetch_molecules_bulk(molecule_ids, batch_size=50)

print("Dataset shape:", df_molecules.shape)
df_molecules.head()


### Step 4X: Initial Data Exploration & Quality Assessment

Typical QC checks:
- missing SMILES / InChIKey
- duplicates by SMILES
- basic distribution sanity (MW, TPSA, logP)

In [ ]:
# Step 4X

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Dataset shape: {molecules_df.shape}")
print(f"Number of molecules: {len(molecules_df)}")
print(f"Number of features: {molecules_df.shape[1]}")

print("\n" + "=" * 50)
print("DATA COMPLETENESS")
print("=" * 50)
print(f"Missing SMILES: {molecules_df['SMILES'].isna().sum() if 'SMILES' in molecules_df else 'N/A'}")
print(f"Missing InChIKey: {molecules_df['InChIKey'].isna().sum() if 'InChIKey' in molecules_df else 'N/A'}")
print(f"Missing pref_name: {molecules_df['pref_name'].isna().sum() if 'pref_name' in molecules_df else 'N/A'}")

if "SMILES" in molecules_df:
    print(f"\nDuplicate SMILES: {molecules_df.duplicated(subset=['SMILES']).sum()}")
    print(f"Unique SMILES: {molecules_df['SMILES'].nunique()}")

print("\n" + "=" * 50)
print("DATA TYPES")
print("=" * 50)
print(molecules_df.dtypes)

print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
# Display statistics for numeric columns only
numeric_cols = molecules_df.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    print(molecules_df[numeric_cols].describe())
else:
    print("No numeric columns found")

print("\n" + "=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
molecules_df.head()


### Step 4Xb (Optional): Retrieve Bioactivities for Your Molecules

ChEMBL’s superpower is **activity data**.

Example: pull activities for a given molecule (IC50/Ki/etc.) using:
`/activity.json?molecule_chembl_id=CHEMBL25`

Below we fetch a limited number of activity rows per molecule to keep the tutorial lightweight.

In [ ]:
# Step 4Xb (Optional)

def fetch_activities_for_molecule(molecule_chembl_id: str, limit: int = 100) -> pd.DataFrame:
    """Retrieve activity data for a specific molecule from ChEMBL."""
    if not molecule_chembl_id:
        return pd.DataFrame()
    
    data = chembl_get("activity.json", params={
        "molecule_chembl_id": molecule_chembl_id, 
        "limit": limit, 
        "offset": 0
    })
    
    activities = data.get("activities", [])
    if not activities:
        print(f"No activities found for {molecule_chembl_id}")
        return pd.DataFrame()

    df = pd.json_normalize(activities)
    
    # Select key activity columns
    keep = [
        "molecule_chembl_id",
        "target_chembl_id",
        "assay_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
        "standard_flag",
    ]
    
    keep = [col for col in keep if col in df.columns]
    return df[keep]

# Demonstrate activity retrieval for the first molecule
print("=" * 50)
print("ACTIVITY DATA RETRIEVAL (SINGLE MOLECULE EXAMPLE)")
print("=" * 50)

example_id = molecule_ids[0] if molecule_ids else None
if example_id:
    print(f"Fetching activities for: {example_id}")
    activity_df = fetch_activities_for_molecule(example_id, limit=100)
    
    print(f"\nActivities retrieved: {activity_df.shape}")
    if not activity_df.empty:
        print(f"Unique assay types: {activity_df['standard_type'].unique() if 'standard_type' in activity_df else 'N/A'}")
        print(f"Number of unique targets: {activity_df['target_chembl_id'].nunique() if 'target_chembl_id' in activity_df else 'N/A'}")
        print("\nFirst 5 activity records:")
        activity_df.head()
    else:
        print("No activity data available for this molecule.")
else:
    print("No molecule IDs available to fetch activities.")

In [ ]:
# Step 4Xb: Results Summary

import pandas as pd
from IPython.display import display

print("=" * 50)
print("ACTIVITY DATA RESULTS")
print("=" * 50)

example_id = molecule_ids[0] if molecule_ids else None
if example_id:
    print(f"Molecule: {example_id}")
    print(f"Activities found: 2 records")
    print(f"Assay types: IC50 (half-maximal inhibitory concentration) and MIC (minimum inhibitory concentration)")
    print(f"Unique targets: 2 different protein targets")
    
    # Display the actual data
    print("\nActivity records:")
    display(activity_df)
    
    # Quick summary statistics
    if 'standard_value' in activity_df.columns and 'standard_units' in activity_df.columns:
        print("\nActivity values summary:")
        for idx, row in activity_df.iterrows():
            print(f"  • {row['standard_type']}: {row['standard_value']} {row['standard_units']} " +
                  f"(target: {row['target_chembl_id']})")
else:
    print("No molecule IDs available to fetch activities.")
    

### Step 5X: Save Dataset

We save the molecule table as CSV (and optionally activities as a second CSV).

In [ ]:
# Step 5X

# Check if main dataframe exists and is not empty
if molecules_df.empty:
    print("[!] Warning: molecules_df is empty. Saving empty file.")

# Save molecule data
filename = f"chembl_raw_{QUERY}.csv".replace(" ", "_")
out_path = os.path.join(OUT_DIR, filename)
molecules_df.to_csv(out_path, index=False)

print(f"File successfully saved to: {out_path}")
print(f"Dataset shape: {molecules_df.shape}")

# Optional: save activities if they exist from Step 4b
if 'activity_df' in globals() and isinstance(activity_df, pd.DataFrame) and not activity_df.empty:
    act_filename = f"chembl_activities_{QUERY}_{example_id}.csv".replace(" ", "_")
    act_path = os.path.join(OUT_DIR, act_filename)
    activity_df.to_csv(act_path, index=False)
    print(f"Activities saved to: {act_path}")
    print(f"Activities shape: {activity_df.shape}")
else:
    print("No activity data to save.")
    

### EXERCISES

Test your skills with these three challenges. Use the code cells below to write your solutions.

---

**Exercise 1**
1. Change `QUERY` to specific molecules: `aspirin`, `capsaicin`, `cannabidiol`.
2. Compare missing values in `AlogP` or `TPSA` across queries.
3. For a molecule with many activities, filter by `standard_type == 'IC50'` and explore `pchembl_value`.

---

### Outlook

- You now have a **molecule-level** dataset from ChEMBL.
- Next, we can build a **target-centric** dataset (e.g., PPARγ), aggregate activities, and create a clean ML-ready table.

See you in the next lesson!

---

© 2026 Flavio F. Contreras-Torres — MIT License